# 채점 함수로 학습시키기 (GRPO)

지금까지 두 가지를 봤습니다.

| | 필요한 것 |
|---|---|
| SFT | 입력–**정답** 쌍 |
| DPO | 좋은 답 / 나쁜 답 **쌍** |

둘 다 **사람이 만든 데이터**가 있어야 합니다. 그런데 어떤 일은 정답을
**코드로 확인**할 수 있습니다. 수학 문제가 그렇습니다 — 답이 맞는지 계산하면 됩니다.

그럴 때는 데이터를 모으는 대신 **채점 함수**를 짜면 됩니다.
모델이 답을 여러 개 만들면, 함수가 점수를 매기고, 높은 쪽으로 밀어냅니다.

이것이 **GRPO**(Group Relative Policy Optimization)입니다.

## 무엇을 하게 되나

1. 한국어 **수학 문제** 데이터를 봅니다
2. 모델에게 **생각하는 형식**을 정해줍니다
3. **채점 함수 두 개**를 만듭니다 — 형식과 정답
4. 학습시키고, 실제로 그 형식으로 답하는지 봅니다

> **GRPO는 가장 비쌉니다.** SFT·DPO는 정답을 한 번 보고 손실을 계산하지만,
> GRPO는 **매 스텝마다 답을 여러 개 생성**하고 각각 채점합니다.
> 생성이 학습 안에 들어가 있어서 스텝당 비용이 몇 배입니다.

## 환경 세팅

In [1]:
%pip install -q -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitli

In [2]:
%pip install -q -U trl peft math-verify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 kB 21.0 MB/s eta 0:00:00


## 1. 데이터 — 답을 검증할 수 있는 것

`numina_math_ko_verifiable_540k`는 한국어 수학 문제 모음입니다.
이름의 **`verifiable`**이 핵심입니다 — 답이 맞는지 **기계가 확인할 수 있다**는 뜻입니다.

| 열 | 내용 |
|---|---|
| `problem` | 문제 |
| `answer` | 정답 (수식) |

GRPO를 쓸 수 있는지 판단하는 기준이 여기 있습니다.
**"이 작업의 정답을 코드로 확인할 수 있는가?"**

| 가능 | 어려움 |
|---|---|
| 수학 — 답이 맞나 | 요약 — 좋은 요약인가 |
| 코드 — 테스트가 통과하나 | 번역 — 자연스러운가 |
| 형식 — 스키마를 지켰나 | 상담 — 공감이 되나 |

오른쪽은 GRPO 대상이 아닙니다. 그쪽은 SFT나 DPO로 갑니다.




In [4]:
from datasets import load_dataset

raw_datasets = load_dataset("OLAIR/numina_math_ko_verifiable_540k")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00002.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00001-of-00002.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/539149 [00:00<?, ? examples/s]

In [5]:
from datasets import DatasetDict

# remove this when done debugging
indices = range(0,1000)
test_indices = range(1000,1050)

dataset_dict = {"train": raw_datasets["train"].select(indices),
                "test": raw_datasets["train"].select(test_indices)}

raw_datasets = DatasetDict(dataset_dict)
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['original', 'reference', 'problem', 'source', 'answer'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['original', 'reference', 'problem', 'source', 'answer'],
        num_rows: 50
    })
})

In [6]:
example = raw_datasets["train"][0]
print(example.keys())

dict_keys(['original', 'reference', 'problem', 'source', 'answer'])


## 토크나이저 불러오기

SFT·DPO와 같은 설정입니다. 생성할 때 템플릿이 한 군데 달라지는데,
그 이유는 아래 생성 절에서 다룹니다.


In [7]:
from transformers import AutoTokenizer

# GRPO 만 구세대 Qwen2.5 를 쓰고 있어 SFT/DPO 와 계열이 갈렸다.
# Qwen3 로 통일한다.
model_id = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token_id is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id

if tokenizer.model_max_length > 100_000:
  tokenizer.model_max_length = 2048

DEFAULT_CHAT_TEMPLATE = "{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"
tokenizer.chat_template = DEFAULT_CHAT_TEMPLATE

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 2. 생각하는 형식을 정해준다

시스템 프롬프트로 **출력 형식**을 못박습니다.

```
<think>
...생각하는 과정...
</think>
<answer>
...최종 답...
</answer>
```

왜 이렇게 나누느냐면, **답만 뽑아내야** 채점할 수 있기 때문입니다.
생각 과정과 답이 뒤섞여 있으면 어디가 답인지 알 수 없습니다.

그리고 생각을 쓰게 하는 것 자체가 정답률을 올립니다.
바로 답하는 것보다 단계를 밟는 편이 낫다는 것은 잘 알려져 있습니다.

**이 형식이 다음 섹션의 채점 함수와 짝입니다.** 프롬프트에서 형식을 요구하고,
채점 함수가 그 형식을 지켰는지 봅니다. 둘 중 하나만 바꾸면 어긋납니다.

In [8]:
import re
import random
from multiprocessing import cpu_count

SYSTEM_PROMPT = """
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>
"""

def apply_chat_template(example):

    return { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': example['problem']}
        ],
        'answer': example['answer']
    }

column_names = list(raw_datasets["train"].features)
raw_datasets = raw_datasets.map(apply_chat_template,
                                num_proc=cpu_count(),
                                remove_columns=column_names,
                                desc="Applying chat template",)

# create the splits
train_dataset = raw_datasets["train"]
eval_dataset = raw_datasets["test"]

for index in random.sample(range(len(raw_datasets["train"])), 3):
  print(f"Sample {index} of the processed training prompt set:\n\n{raw_datasets['train'][index]['prompt']}")
  print(f"Sample {index} of the processed training answer set:\n\n{raw_datasets['train'][index]['answer']}")

Applying chat template (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

Applying chat template (num_proc=8):   0%|          | 0/50 [00:00<?, ? examples/s]

Sample 255 of the processed training prompt set:

[{'content': '\nRespond in the following format:\n<think>\n...\n</think>\n<answer>\n...\n</answer>\n', 'role': 'system'}, {'content': '$3^{20}$를 5로 나눈 나머지는 __________입니다.', 'role': 'user'}]
Sample 255 of the processed training answer set:

1
Sample 377 of the processed training prompt set:

[{'content': '\nRespond in the following format:\n<think>\n...\n</think>\n<answer>\n...\n</answer>\n', 'role': 'system'}, {'content': '발표회에서 학생들이 차례로 발표를 합니다. 은정이는 뒤에서 6번째 발표자이며, 은정이 앞에 있는 7명의 학생이 발표를 합니다. 총 몇 명의 학생이 발표를 하고 있나요?', 'role': 'user'}]
Sample 377 of the processed training answer set:

13
Sample 964 of the processed training prompt set:

[{'content': '\nRespond in the following format:\n<think>\n...\n</think>\n<answer>\n...\n</answer>\n', 'role': 'system'}, {'content': '양의 정수 $a$의 각 자리 숫자의 합이 6이면, $a$를 "좋은 수"라고 합니다 (예를 들어, 6, 24, 2013은 모두 "좋은 수"입니다). 모든 "좋은 수"를 오름차순으로 나열하여 $a_1$, $a_2$, $a_3$, …로 표시했을 때, 만약 $a_n = 2013$이라면 $n =$ (\u3000\u3

In [9]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['answer', 'prompt'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['answer', 'prompt'],
        num_rows: 50
    })
})

## 3. 채점 함수 — 이 실습의 핵심

GRPO 에는 **정답 라벨이 없습니다.** 대신 **답을 채점하는 함수**가 있습니다.

두 개를 씁니다.

| 함수 | 무엇을 보나 | 왜 |
|---|---|---|
| `format_reward` | `<think>…</think><answer>…</answer>` 형식을 지켰나 | 형식이 깨지면 답을 꺼낼 수 없다 |
| `accuracy_reward` | 답이 정답과 **수학적으로 같은가** | 이게 진짜 목표다 |

### 두 번째가 중요합니다

문자열로 비교하면 `1/2`와 `0.5`를 다른 답으로 봅니다. 둘 다 맞는데도요.

`math_verify`는 수식을 **파싱해서** 비교합니다. 표현이 달라도 값이 같으면
맞다고 판정합니다. 그래서 채점이 실제 정답률에 가까워집니다.

**채점 함수의 품질이 곧 학습의 품질입니다.** 함수가 엉성하면 모델은
그 엉성함을 파고듭니다 — 형식만 그럴듯하게 맞추고 답은 틀리는 식으로요.
`format_reward` 하나만 썼다면 정확히 그렇게 됐을 겁니다.

> **이것이 GRPO와 PPO의 차이이기도 합니다.** PPO는 사람의 선호를 배운
> **보상 모델**(또 하나의 신경망)을 씁니다. GRPO는 그냥 **함수**를 씁니다.
> 채점을 코드로 짤 수 있으면 모델 하나를 통째로 아낍니다.

In [10]:
import re

def format_reward(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    completion_contents = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, content) for content in completion_contents]
    rewards_list = [1.0 if match else 0.0 for match in matches]

    return [1.0 if match else 0.0 for match in matches]


In [11]:
from math_verify import LatexExtractionConfig, parse, verify

def accuracy_reward(completions, **kwargs):
    """Reward function that checks if the completion is the same as the ground truth."""
    solutions = kwargs['answer']
    completion_contents = [completion[0]["content"] for completion in completions]
    rewards = []
    for content, solution in zip(completion_contents, solutions):
        gold_parsed = parse(solution, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        answer_parsed = parse(content, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        if len(gold_parsed) != 0:
            try:
                rewards.append(float(verify(answer_parsed, gold_parsed)))
            except Exception:
                rewards.append(0.0)
        else:
            rewards.append(1.0)
    return rewards

## 4. 학습 설정

`num_generations=4`가 GRPO의 정의에 해당하는 값입니다.

**문제 하나에 답을 4개 만듭니다.** 그리고 그 4개를 **서로 비교**합니다 —
평균보다 잘한 답은 확률을 올리고, 못한 답은 내립니다.
이름의 **G**roup **R**elative가 이 뜻입니다. 그룹 안에서 상대적으로 평가합니다.

절대 점수가 아니라 상대 비교라서, 별도의 기준선 모델이 필요 없습니다.
그룹의 평균이 기준선 역할을 합니다.

| 값 | 뜻 |
|---|---|
| `num_generations=4` | **그룹 크기.** 크면 비교가 안정되고 그만큼 느려진다 |
| `max_completion_length=128` | 답 길이 상한. 생성이 학습 안에 있어 시간에 직결된다 |
| `gradient_accumulation_steps=16` | 메모리를 아끼려고 기울기를 모았다 갱신 |
| LoRA `r=8` | SFT의 64보다 작다. 생성 부담이 커서 가볍게 간다 |

> 학습 로그의 `reward`가 올라가는지 보세요. 두 채점 함수의 합입니다.
> `format_reward`가 먼저 1.0에 가까워지고, `accuracy_reward`가 천천히 따라옵니다 —
> 형식을 지키는 것이 답을 맞히는 것보다 쉽기 때문입니다.

In [12]:
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [13]:
from trl import GRPOTrainer, GRPOConfig

output_dir = 'data/grpo_model'

training_args = GRPOConfig(
    output_dir=output_dir,
    learning_rate=1e-4,
    remove_unused_columns=False,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    bf16=True,
    max_completion_length=128,
    num_generations=4,
    report_to=["tensorboard"],
    logging_steps=1,
    push_to_hub=False,
    save_strategy="steps",
    save_steps=10,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward, accuracy_reward],
    args=training_args,
    train_dataset=train_dataset,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


## 학습

시간이 꽤 걸립니다 — 매 스텝 답을 4개씩 **생성하면서** 학습하기 때문입니다.

In [14]:
train_result = trainer.train()

Step,Training Loss
1,0.000000
2,0.000000
3,-0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


In [15]:
trainer.model.save_pretrained(output_dir)
trainer.processing_class.save_pretrained(output_dir)

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


('data/test_model/tokenizer_config.json',
 'data/test_model/special_tokens_map.json',
 'data/test_model/vocab.json',
 'data/test_model/merges.txt',
 'data/test_model/added_tokens.json',
 'data/test_model/tokenizer.json')

## 5. 써보기 — 템플릿이 학습 때와 다릅니다

아래 셀에서 챗 템플릿을 다시 넣는데, **학습 때와 한 군데가 다릅니다.**

```
학습:  ... <|assistant|>
생성:  ... <|assistant|>
       <think>              ← 이 줄이 추가됨
```

모델이 답을 시작하는 자리에 **`<think>`를 미리 넣어줍니다.**
그러면 모델은 생각부터 이어 쓸 수밖에 없습니다. 형식을 강제하는 셈입니다.

이런 기법을 **프리필**(prefill)이라고 합니다. 학습으로 형식을 가르치되,
생성할 때 한 번 더 못을 박는 것입니다.

출력에 `<think>`와 `<answer>`가 제대로 나오는지 보세요.
나오지 않으면 학습이 부족한 것이고, 실습 규모에서는 흔히 그렇습니다.

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

output_dir = 'data/grpo_model'
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")

In [17]:
tokenizer.chat_template = "{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>\n<think>' }}\n{% endif %}\n{% endfor %}"

In [21]:
import torch

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    },
    {"role": "user", "content": "철수가 가진 사탕 2개와 영희가 가진 사탕 3개를 합치면 몇개?"},
]

# prepare the messages for the model
# transformers 5 에서 apply_chat_template 은 BatchEncoding 을 돌려준다.
# 예전처럼 generate(input_ids=...) 로 넘기면 AttributeError 가 난다.
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
).to("cuda")

# inference
outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.6,
        top_p=0.95
)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

<|system|>

Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>

<|user|>
철수가 가진 사탕 2개와 영희가 가진 사탕 3개를 합치면 몇개?
<|assistant|>
<think>
2 + 3 = 5
</think>
<answer>
5
</answer>


## 마무리

채점 함수로 학습시키는 방법을 봤습니다.

- GRPO는 정답 라벨이 아니라 **채점 함수**를 씁니다
- 그래서 **답을 코드로 확인할 수 있는 일**에만 쓸 수 있습니다
- 문제 하나에 답을 여러 개 만들어 **서로 비교**합니다 (Group Relative)
- **채점 함수의 품질이 곧 학습의 품질**입니다. 엉성하면 모델이 파고듭니다
- 생성이 학습 안에 있어 **가장 비쌉니다**

### 세 가지를 다 봤습니다

| 방법 | 필요한 것 | 비용 |
|---|---|---|
| SFT | 입력–정답 쌍 | 보통 |
| DPO / ORPO | 좋은 답 / 나쁜 답 쌍 | 보통 |
| GRPO | **채점 함수** | 가장 비쌈 |

무엇을 고를지는 **가진 데이터가 정합니다.** 방법을 먼저 정하고 데이터를
맞추는 것이 아닙니다. 그리고 그 위에 **학습하지 않는 선택지**가 있습니다 —
2일차에 본 프롬프트 최적화입니다.

**프롬프트로 되면 학습하지 않습니다.** 그것이 이 과정 전체의 결론입니다.